# Model Selection and Robustness using k-Fold Cross Validation(CV)

In [ ]:
# Required libraries
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV, KFold, LeaveOneOut, StratifiedKFold
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from time import time

In [ ]:
from sklearnex import patch_sklearn
patch_sklearn()

Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


In [ ]:
# Importing the data

data = load_breast_cancer()
X = data.data
y = data.target

In [ ]:
# analyse the data

print(data.DESCR)

.. _breast_cancer_dataset:

Breast cancer wisconsin (diagnostic) dataset
--------------------------------------------

**Data Set Characteristics:**

:Number of Instances: 569

:Number of Attributes: 30 numeric, predictive attributes and the class

:Attribute Information:
    - radius (mean of distances from center to points on the perimeter)
    - texture (standard deviation of gray-scale values)
    - perimeter
    - area
    - smoothness (local variation in radius lengths)
    - compactness (perimeter^2 / area - 1.0)
    - concavity (severity of concave portions of the contour)
    - concave points (number of concave portions of the contour)
    - symmetry
    - fractal dimension ("coastline approximation" - 1)

    The mean, standard error, and "worst" or largest (mean of the three
    worst/largest values) of these features were computed for each image,
    resulting in 30 features.  For instance, field 0 is Mean Radius, field
    10 is Radius SE, field 20 is Worst Radius.

    - 

In [ ]:
N = len(y)
print(f"Total number of samples (N): {N}")

Total number of samples (N): 569


In [ ]:
# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
param_grid = {
    'alpha' : [1e-5, 1e-3, 1e-2, 1e-1, 1], # Regularization paramerter
    'penalty' : ['l1', 'l2', 'elasticnet'], # Regularization type
    'learning_rate' : ['constant', 'optimal', 'invscaling'],
    'eta0' : [1e-2, 1e-1, 1], # Initial learning rate
    'early_stopping' : [True, False]
}

In [ ]:
random_state = 42

In [ ]:
base_estimator = SGDClassifier(random_state=random_state, loss = 'log_loss', max_iter = 1000, tol = 1e-3)

In [ ]:
def run_cv_experiment(k, X, y, param_grid, estimator, n_repeats=1):
    """Runs the GridSearchCV for a specified k and number of repeats."""
    results = []

    # Define the CV strategy
    if k == N:
        cv_strategy = LeaveOneOut() # LOOCV
        k_label = 'LOOCV (k=N)'
    elif k == 2:
        cv_strategy = KFold(n_splits=k, shuffle=True, random_state=42) # For fixed k, we use KFold
        k_label = f'k={k}'
    else:
        cv_strategy = KFold(n_splits=k, shuffle=True, random_state=42)
        k_label = f'k={k}'

    for repeat in range(n_repeats):
        start_time = time()

        # Use a different random_state for the CV split in each repeat (if not fixed by KFold's random_state)
        # Note: GridSearchCV handles the model random_state via the grid, but we need to ensure fold splitting varies
        if k != N:
            # Re-initialize KFold for randomization across repeats (critical for k=2 analysis)
            cv_strategy = StratifiedKFold(n_splits=k, shuffle=True, random_state=repeat)

        grid_search = GridSearchCV(
            estimator=estimator,
            param_grid=param_grid,
            cv=cv_strategy,
            scoring='accuracy',
            n_jobs=-1
        )

        # NOTE: Fit modifies the estimator, which is fine since we re-initialize the best_estimator
        grid_search.fit(X, y)
        end_time = time()

        results.append({
            'k': k_label,
            'repeat': repeat + 1,
            'time_seconds': end_time - start_time,
            'best_score': grid_search.best_score_,
            'best_params': grid_search.best_params_
        })

    return results

In [ ]:
# Scenario 2: Standard (k=5) - Optimal Hyperparameters
print("\n--- Running Scenario 2 (k=5) ---")
results_k5 = run_cv_experiment(k=5, X=X_scaled, y=y, param_grid=param_grid, estimator=base_estimator, n_repeats=1)


--- Running Scenario 2 (k=5) ---


In [ ]:
# Scenarios 1, 2, 3: Computation Time Comparison
print("\n--- Running Scenarios for Computation Time Comparison ---")
results_k2_time = run_cv_experiment(k=2, X=X_scaled, y=y, param_grid=param_grid, estimator=base_estimator, n_repeats=1)
results_kN_time = run_cv_experiment(k=N, X=X_scaled, y=y, param_grid=param_grid, estimator=base_estimator, n_repeats=1)


--- Running Scenarios for Computation Time Comparison ---


In [ ]:
# Scenario 1 & 3: Variability and Robustness (5 Repeats)
print("\n--- Running Scenarios for Variability (5 Repeats) ---")
results_k2_var = run_cv_experiment(k=2, X=X_scaled, y=y, param_grid=param_grid, estimator=base_estimator, n_repeats=5)
results_kN_var = run_cv_experiment(k=N, X=X_scaled, y=y, param_grid=param_grid, estimator=base_estimator, n_repeats=5)


--- Running Scenarios for Variability (5 Repeats) ---


In [ ]:
# --- 5. Display and Analyze Results ---
all_results = results_k5 + results_k2_time + results_kN_time + results_k2_var + results_kN_var
results_df = pd.DataFrame(all_results)
print("\n--- Full Results Table ---")
print(results_df[['k', 'repeat', 'time_seconds', 'best_score', 'best_params']])

In [ ]:
# Summary of Variability
var_summary = results_df[results_df['repeat'] > 0].groupby('k').agg(
    Mean_Score=('best_score', 'mean'),
    Std_Dev_Score=('best_score', 'std'),
    Mean_Time=('time_seconds', 'mean'),
    Repeats=('repeat', 'count')
)
print("\n--- Variability and Time Summary ---")
print(var_summary.fillna({'Std_Dev_Score': 0})) # LOOCV will likely have 0 std dev if time is consistent